In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.float_format", "{:.1f}".format)
pd.set_option("display.max_columns", 20)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 100, "font.size": 11})

In [ ]:
url = "https://share.borza.cc/"

SOURCES = {"details": "ingatlan-details.parquet", "listings": "listings.parquet"}


def load_dataset(name: str) -> pd.DataFrame:
    fp = Path("data", f"ingatlan-{name}.parquet")
    if fp.exists():
        return pd.read_parquet(fp)
    df = pd.read_parquet(url + fp.name)
    df.to_parquet(fp)
    return df

details = load_dataset("details")
listings = load_dataset("listings")

print(f"details:  {details.shape[0]:,} rows x {details.shape[1]} columns")
print(f"listings: {listings.shape[0]:,} rows x {listings.shape[1]} columns")
details.head(3)

In [ ]:
listings.head(3)

## 1. `.pipe` and `.str`

- **`.str.extract()`** — pull the Budapest district (`IX. kerület`) from the free-text `loc` field
- **`.str.replace()` + `pd.to_numeric`** — parse `'2 fél'` → `2.5`, `'3'` → `3.0`
- **`.str.extract()` again** — classify the street type (`utca`, `út`, `tér`, ...)
- **`pipe`** — compose all transformations without intermediate variables

In [ ]:
def extract_district(df: pd.DataFrame) -> pd.DataFrame:
    """Extract Budapest district label (e.g. 'VIII. kerület') from the loc field."""
    return df.assign(
        district=df["loc"].str.extract(r"\b([IVX]+\.\s*kerület)\b", expand=False)
    )


def parse_rooms(df: pd.DataFrame) -> pd.DataFrame:
    """Convert '2 fél' -> 2.5, '3' -> 3.0; unparseable -> NaN."""
    return df.assign(
        rooms_num=(
            df["rooms"]
            .str.replace(" fél", ".5", regex=False)
            .pipe(pd.to_numeric, errors="coerce")
        )
    )


def add_price_per_m2(df: pd.DataFrame) -> pd.DataFrame:
    return df.assign(price_per_m2=(df["price"] / df["m2"]).round(1))


def flag_balcony(df: pd.DataFrame) -> pd.DataFrame:
    return df.assign(has_balcony=df["balcony"].notna())


def extract_street_type(df: pd.DataFrame) -> pd.DataFrame:
    """Extract street type as the last word of the loc field (.str chaining showcase)."""
    return df.assign(
        street_type=df["loc"].str.lower().str.split().str[-1].str.rstrip(".,")
    )


enriched = (
    details.pipe(extract_district)
    .pipe(parse_rooms)
    .pipe(add_price_per_m2)
    .pipe(flag_balcony)
    .pipe(extract_street_type)
)

enriched[
    [
        "loc",
        "district",
        "street_type",
        "rooms",
        "rooms_num",
        "price_per_m2",
        "has_balcony",
    ]
].head(6)

In [ ]:
enriched["district"].value_counts()

## 2. `value_counts`

In [ ]:
top_cities = (
    enriched["city"]
    .value_counts()
    .head(20)
    .rename_axis("city")
    .reset_index(name="listings")
    .assign(share_pct=lambda df: (df["listings"] / df["listings"].sum() * 100).round(2))
)

(
    top_cities.style.background_gradient(cmap="Blues", subset=["listings"])
    .bar(subset=["share_pct"], color="#aec6cf", vmin=0)
    .format({"listings": "{:,}", "share_pct": "{:.2f}%"})
    .set_caption("Top 20 cities by listing count")
)

In [ ]:
room_counts = (
    enriched["rooms_num"]
    .dropna()
    .value_counts()
    .sort_index()
    .loc[lambda s: s.index <= 7]
)

balcony_share = (
    enriched["has_balcony"]
    .map({True: "With balcony", False: "No balcony"})
    .value_counts()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(
    room_counts.index.astype(str),
    room_counts.values,
    color="steelblue",
    width=0.6,
    alpha=0.85,
)
axes[0].set_xlabel("Room count")
axes[0].set_ylabel("Listings")
axes[0].set_title("Room count distribution")

axes[1].pie(
    balcony_share,
    labels=balcony_share.index,
    autopct="%1.1f%%",
    colors=["#4c7b8a", "#c8c8c8"],
    startangle=90,
)
axes[1].set_title("Balcony presence")

plt.tight_layout()
plt.show()

## 3. `groupby + agg`

In [ ]:
def city_price_summary(df: pd.DataFrame, min_listings: int = 200) -> pd.DataFrame:
    return (
        df.groupby("city")["price_per_m2"]
        .agg(
            listings="count",
            median="median",
            mean="mean",
            p25=lambda x: x.quantile(0.25),
            p75=lambda x: x.quantile(0.75),
            std="std",
        )
        .query("listings >= @min_listings")
        .sort_values("median", ascending=False)
        .round(0)
        .astype({"listings": int})
        .head(20)
    )


city_stats = enriched.pipe(city_price_summary)

(
    city_stats.style.background_gradient(cmap="RdYlGn_r", subset=["median", "mean"])
    .background_gradient(cmap="Blues", subset=["listings"])
    .format(
        {
            "listings": "{:,}",
            "median": "{:.0f}",
            "mean": "{:.0f}",
            "p25": "{:.0f}",
            "p75": "{:.0f}",
            "std": "{:.0f}",
        }
    )
    .set_caption(
        "Price per m\u00b2 (eFt) by city \u2014 top 20 most expensive (min. 200 listings)"
    )
)

In [ ]:
district_medians = (
    enriched.loc[enriched["district"].notna()]
    .groupby("district")["price_per_m2"]
    .median()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(district_medians))[::-1])
ax.barh(district_medians.index, district_medians.values, color=colors, alpha=0.88)
ax.set_xlabel("Median price per m\u00b2 (eFt)")
ax.set_title("Budapest districts: median price per m\u00b2")
plt.tight_layout()
plt.show()

## 4. Group Profiles with `groupby + apply`

In [ ]:
def district_room_profile(group: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "listings": len(group),
            "median_ppm2": group["price_per_m2"].median(),
            "median_m2": group["m2"].median(),
            "balcony_pct": group["has_balcony"].mean() * 100,
        }
    )


district_profiles = (
    enriched.loc[enriched["district"].notna() & enriched["rooms_num"].between(1, 5)]
    .groupby(["district", "rooms_num"])
    .apply(district_room_profile, include_groups=False)
    .round(1)
    .astype({"listings": int})
)

district_profiles.head(12)

In [ ]:
# Show the most expensive district profiles
(
    district_profiles.query("listings >= 200")
    .nlargest(15, "median_ppm2")
    .style.background_gradient(cmap="RdYlGn_r", subset=["median_ppm2"])
    .background_gradient(cmap="Greens", subset=["balcony_pct"])
    .background_gradient(cmap="Blues", subset=["listings"])
    .format(
        {
            "listings": "{:,}",
            "median_ppm2": "{:.0f}",
            "median_m2": "{:.0f}",
            "balcony_pct": "{:.1f}%",
        }
    )
    .set_caption("Top 15 district x room-count combinations by median price/m\u00b2")
)

## 5. Cross-tabulation with `pivot_table`

A pivot table as a price heatmap: room count on the index, district on the columns.

In [ ]:
def build_ppm2_pivot(df: pd.DataFrame) -> pd.DataFrame:
    top_districts = (
        df.loc[df["district"].notna()]
        .groupby("district")["price_per_m2"]
        .median()
        .nlargest(10)
        .index
    )
    return pd.pivot_table(
        df.loc[df["district"].isin(top_districts) & df["rooms_num"].between(1, 3)],
        values="price_per_m2",
        index="rooms_num",
        columns="district",
        aggfunc="median",
    ).round(0)


ppm2_pivot = enriched.pipe(build_ppm2_pivot)

(
    ppm2_pivot.style.background_gradient(cmap="RdYlGn_r", axis=None)
    .format("{:.0f}", na_rep="\u2014")
    .set_caption(
        "Median price per m\u00b2 (eFt) \u2014 room count \u00d7 district (top 10 districts)"
    )
)

In [ ]:
# Price/m2 distribution by room count — Budapest, clipped at 99th percentile
bp_data = enriched.loc[
    (enriched["city"] == "Budapest")
    & enriched["rooms_num"].between(1, 5)
    & (enriched["price_per_m2"] < enriched["price_per_m2"].quantile(0.99))
]

room_values = sorted(bp_data["rooms_num"].dropna().unique())
groups = [
    bp_data.loc[bp_data["rooms_num"] == r, "price_per_m2"].values for r in room_values
]

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(
    groups,
    labels=[str(r) for r in room_values],
    patch_artist=True,
    medianprops=dict(color="#e84855", lw=2),
)
for patch in bp["boxes"]:
    patch.set_facecolor("#aec6cf")
    patch.set_alpha(0.8)
ax.set_xlabel("Room count")
ax.set_ylabel("Price per m\u00b2 (eFt)")
ax.set_title("Price/m\u00b2 distribution by room count \u2014 Budapest")
plt.tight_layout()
plt.show()

## 6. Enrichment via `merge`

Join the static property details with a summary of their listing history:
first price, latest price, total observations, and the percentage price change.

In [ ]:
def summarise_listing_history(listings: pd.DataFrame) -> pd.DataFrame:
    """First/last price and observation count per property id."""
    return (
        listings.sort_values("day")
        .groupby("id")
        .agg(
            first_price=("price", "first"),
            last_price=("price", "last"),
            first_seen=("day", "first"),
            last_seen=("day", "last"),
            n_obs=("price", "count"),
        )
        .reset_index()
        .assign(
            price_change_pct=lambda df: (
                (df["last_price"] - df["first_price"]) / df["first_price"] * 100
            ).round(1)
        )
    )


history = summarise_listing_history(listings)

enriched_full = enriched.merge(history, on="id", how="left")

enriched_full[
    [
        "city",
        "m2",
        "rooms_num",
        "price_per_m2",
        "first_seen",
        "last_seen",
        "n_obs",
        "price_change_pct",
    ]
].head(5)

In [ ]:
(
    enriched_full.query("n_obs >= 5")
    .nlargest(15, "n_obs")[
        [
            "loc",
            "city",
            "m2",
            "rooms_num",
            "price_per_m2",
            "price",
            "n_obs",
            "price_change_pct",
            "first_seen",
            "last_seen",
        ]
    ]
    .style.background_gradient(cmap="Blues", subset=["n_obs"])
    .background_gradient(cmap="RdYlGn", subset=["price_change_pct"])
    .format(
        {
            "price_per_m2": "{:.0f}",
            "rooms_num": "{:.1f}",
            "n_obs": "{:.0f}",
            "price_change_pct": "{:+.1f}%",
        }
    )
    .set_caption("Properties with most price observations")
)

In [ ]:
# Price change distribution — discount vs appreciation
changes = (
    enriched_full.loc[lambda df: (df["n_obs"] >= 3) & (df["price_change_pct"] != 0)][
        "price_change_pct"
    ]
    .dropna()
    .clip(-50, 50)
)

n_unchanged = (
    enriched_full.loc[lambda df: df["n_obs"] >= 3, "price_change_pct"]
    .pipe(lambda s: s == 0)
    .sum()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(
    changes, bins=80, color="steelblue", alpha=0.8, edgecolor="white", linewidth=0.3
)
ax.axvline(0, color="#e84855", lw=2, linestyle="--", label="No change")
ax.axvline(
    changes.median(),
    color="orange",
    lw=2,
    linestyle="--",
    label=f"Median: {changes.median():+.1f}%",
)
ax.set_xlabel("Price change from first to last observation (%)")
ax.set_ylabel("Properties")
ax.set_title(
    f"Distribution of price changes (clipped to \u00b150%) {n_unchanged} unchanged"
)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Market Price Trend with `rolling`

Aggregate to a daily median, then apply 30- and 90-day rolling averages to smooth out noise.

In [ ]:
def market_price_trend(listings: pd.DataFrame) -> pd.DataFrame:
    """Daily median listing price with 30- and 90-day rolling averages."""
    return (
        listings.groupby("day")["price"]
        .median()
        .reset_index(name="daily_median")
        .sort_values("day")
        .assign(
            rolling_30d=lambda df: df["daily_median"].rolling(30, min_periods=7).mean(),
            rolling_90d=lambda df: df["daily_median"]
            .rolling(90, min_periods=30)
            .mean(),
        )
    )


trend = market_price_trend(listings)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(
    trend["day"],
    trend["daily_median"],
    alpha=0.2,
    color="#2e86ab",
    lw=1,
    label="Daily median",
)
ax.plot(
    trend["day"],
    trend["rolling_30d"],
    color="#2e86ab",
    lw=2,
    label="30-day rolling avg",
)
ax.plot(
    trend["day"],
    trend["rolling_90d"],
    color="#e84855",
    lw=2.5,
    label="90-day rolling avg",
)
ax.set_xlabel("Date")
ax.set_ylabel("Median listing price (eFt)")
ax.set_title("Hungarian real estate market: price trend")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Per-City Trends with `groupby + apply` + `rolling`

Merge listings with city information, compute a daily median per city,
then use `groupby + apply` to add a rolling average inside each city group.

In [ ]:
def add_rolling_30d(group: pd.DataFrame) -> pd.DataFrame:
    """Add 30-day rolling mean price column within a single city group."""
    return group.assign(rolling_30d=group["price"].rolling(30, min_periods=5).mean())


top5_cities = details["city"].value_counts().head(5).index
top5_ids = set(details.loc[details["city"].isin(top5_cities), "id"])

city_trends = (
    listings.loc[listings["id"].isin(top5_ids)]
    .merge(details[["id", "city"]], on="id")
    .groupby(["city", "day"])["price"]
    .median()
    .reset_index()
    .sort_values(["city", "day"])
    .groupby("city", group_keys=True)
    .apply(add_rolling_30d)
)

fig, ax = plt.subplots(figsize=(13, 5))
for city, group in city_trends.groupby("city"):
    ax.plot(group["day"], group["rolling_30d"], label=city, lw=2)
ax.set_xlabel("Date")
ax.set_ylabel("Rolling 30-day median price (eFt)")
ax.set_title("Price trend by city \u2014 top 5 by listing volume")
ax.legend(title="City")
plt.tight_layout()
plt.show()